# Logging & Debugging

The `logging` module gives you production-grade observability: structured messages, severity levels, multiple simultaneous destinations, and hierarchical loggers that let library and application code coexist cleanly. Pair it with `breakpoint()` for interactive debugging and the `traceback` module for programmatic error introspection.

**What's inside:** log levels, `basicConfig`, named loggers, handlers, formatters, dual console+file logging, `breakpoint()`/`pdb`, `traceback`, and `warnings`.

**Learn more:** [logging](https://docs.python.org/3/library/logging.html) · [logging HOWTO](https://docs.python.org/3/howto/logging.html) · [pdb](https://docs.python.org/3/library/pdb.html)

## 1. Log levels

In [ ]:
import logging

# Five standard levels, lowest to highest severity
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')

logging.debug('fine-grained diagnostic info')
logging.info('normal operational message')
logging.warning('something unexpected, but not an error')
logging.error('an error occurred; execution can continue')
logging.critical('serious failure; program may not be able to continue')

## 2. Named loggers: the idiomatic pattern

In [ ]:
import logging

# Best practice: one logger per module, named after it
logger = logging.getLogger(__name__)    # '__main__' at top level
logger.setLevel(logging.DEBUG)

# Handler: where messages go
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)

# Formatter: how messages look
fmt = logging.Formatter('%(asctime)s  %(name)s  %(levelname)-8s  %(message)s')
handler.setFormatter(fmt)

logger.addHandler(handler)
logger.propagate = False   # don't also send to root logger

logger.info('logger configured')
logger.warning('this is a warning')

## 3. Logger hierarchy

In [ ]:
import logging

# Loggers form a dotted-name tree; children propagate to parents
logging.basicConfig(level=logging.DEBUG, format='%(name)s  %(levelname)s  %(message)s')

root   = logging.getLogger()           # root logger
app    = logging.getLogger('myapp')    # child of root
db     = logging.getLogger('myapp.db') # child of myapp

app.info('application started')
db.debug('connected to database')      # propagates up to myapp → root

# Silencing a noisy third-party library:
logging.getLogger('urllib3').setLevel(logging.WARNING)

## 4. Multiple handlers: console + file simultaneously

In [ ]:
import logging, tempfile
from pathlib import Path

tmp_log = Path(tempfile.mkdtemp()) / 'app.log'

logger = logging.getLogger('dual')
logger.setLevel(logging.DEBUG)
logger.propagate = False

# Console: show WARNING and above
console = logging.StreamHandler()
console.setLevel(logging.WARNING)
console.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))

# File: capture everything
filer = logging.FileHandler(tmp_log, encoding='utf-8')
filer.setLevel(logging.DEBUG)
filer.setFormatter(logging.Formatter('%(asctime)s %(levelname)-8s %(message)s'))

logger.addHandler(console)
logger.addHandler(filer)

logger.debug('debug detail (file only)')
logger.info('info (file only)')
logger.warning('warning (console AND file)')
logger.error('error (console AND file)')

print('\n--- file contents ---')
print(tmp_log.read_text())

## 5. Logging exceptions with exc_info

In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')
logger = logging.getLogger('exceptions')

try:
    result = 1 / 0
except ZeroDivisionError:
    # exc_info=True appends the full traceback to the log record
    logger.error('division failed', exc_info=True)

# Shorthand: logger.exception() implies exc_info=True
try:
    int('not a number')
except ValueError:
    logger.exception('conversion error')

## 6. breakpoint() and pdb

In [ ]:
# breakpoint() (Python 3.7+) drops into pdb, the interactive debugger.
# Useful commands once inside pdb:
#   n (next):      step over one line
#   s (step):      step into a function call
#   c (continue):  run until next breakpoint
#   p expr:       print expression
#   l:             list surrounding source
#   q:             quit
#   h:             help

def buggy_function(items):
    total = 0
    for item in items:
        # Uncomment the next line to pause here during execution:
        # breakpoint()
        total += item
    return total

print(buggy_function([1, 2, 3]))   # 6

In [ ]:
import pdb

# Programmatic breakpoint at a specific line:
def compute(x, y):
    result = x * y
    # pdb.set_trace()    # equivalent to breakpoint() for older Python
    return result + 1

print(compute(3, 4))   # 13

## 7. The traceback module

In [ ]:
import traceback

def c(): return 1 / 0
def b(): return c()
def a(): return b()

try:
    a()
except ZeroDivisionError:
    # format_exc() gives the full traceback as a string
    tb_str = traceback.format_exc()
    print(tb_str)
    print('Last line:', tb_str.strip().splitlines()[-1])

In [ ]:
import traceback, sys

def risky():
    raise ValueError('something went wrong')

try:
    risky()
except ValueError:
    exc_type, exc_value, exc_tb = sys.exc_info()
    # walk the traceback frames
    for frame in traceback.extract_tb(exc_tb):
        print(f'{frame.filename}:{frame.lineno}  {frame.name}  →  {frame.line}')

## 8. warnings module

In [ ]:
import warnings

# Emit a deprecation warning from library code
def old_api(x):
    warnings.warn(
        'old_api() is deprecated; use new_api() instead',
        DeprecationWarning,
        stacklevel=2,   # points at the caller, not this function
    )
    return x * 2

old_api(5)

# Treat all DeprecationWarnings as errors (useful in tests)
warnings.filterwarnings('error', category=DeprecationWarning)
try:
    old_api(5)
except DeprecationWarning as e:
    print(f'caught: {e}')